# Multi-agent systems — interactive companion

Companion to [Post 5a: Multi-Agent Systems](../posts/05a-multi-agent-systems.qmd).

When do more agents help? This notebook lets you watch the Condorcet jury
theorem in action, see correlated errors cap voting, watch debate herd under
conformity, and feel pipelines compound errors.

**You'll do (~20 minutes):**
1. Confirm the Condorcet jury theorem (independent agents → certainty).
2. Watch correlation cap voting at 1 − ρ(1−p).
3. See debate help (moderate conformity) and herd (high conformity).
4. Compound a pipeline: reliability = p^L.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["figure.dpi"] = 110

from nano_agents.multiagent import (
    Committee, committee_accuracy, condorcet_accuracy,
    correlated_vote_ceiling, debate, debate_accuracy, pipeline_accuracy,
)

## 1. The Condorcet jury theorem

Independent agents each correct with probability p, majority vote. Above
chance (p > 0.5) the committee votes its way toward certainty; below chance,
toward certain failure.

In [ ]:
sizes = [1, 3, 5, 9, 15, 21, 31, 51]
for p, c in [(0.65, "#55a467"), (0.55, "#3a7ebf"), (0.45, "#c44e52")]:
    plt.plot(sizes, [condorcet_accuracy(p, n) for n in sizes], "o-",
             color=c, label=f"p={p}")
plt.axhline(0.5, color="#bbb", ls=":")
plt.xlabel("committee size"); plt.ylabel("majority-vote accuracy")
plt.legend(); plt.grid(alpha=0.3); plt.show()

# A 3-agent committee at p=0.6:
print("3 agents at p=0.6 ->", round(condorcet_accuracy(0.6, 3), 3))
print("21 agents at p=0.6 ->", round(condorcet_accuracy(0.6, 21), 3))

### Try this
- Set `p=0.49`. Even a tiny disadvantage drives a big committee to near-zero
  accuracy — the theorem amplifies whatever tendency exists.

## 2. Correlation caps voting

The theorem assumes *independent* errors. With lockstep correlation ρ (agents
sharing blind spots), majority-vote accuracy is capped at 1 − ρ(1−p), no
matter how many agents you add.

In [ ]:
p = 0.6
sizes = [1, 3, 5, 9, 15, 25, 41]
for rho, c in [(0.0, "#55a467"), (0.3, "#3a7ebf"), (0.9, "#c44e52")]:
    ys = [committee_accuracy(Committee(n, 2, p, rho, 0), 5000, 1) for n in sizes]
    plt.plot(sizes, ys, "o-", color=c, label=f"ρ={rho}")
    if rho > 0:
        plt.axhline(correlated_vote_ceiling(p, rho), color=c, ls=":", alpha=0.6)
plt.xlabel("committee size"); plt.ylabel("accuracy"); plt.legend()
plt.grid(alpha=0.3); plt.show()

# Effective votes: 40 agents at rho=0.1 are worth...
print("n_eff(40, 0.1) =", round(40 / (1 + 39*0.1), 1))
print("n_eff(40, 0.5) =", round(40 / (1 + 39*0.5), 1))

**Diversity is the resource.** Forty correlated agents at ρ=0.1 carry the
information of about eight independent ones. Engineer diversity (different
models, prompts, tools), not headcount.

## 3. Debate: help vs herding

Debate lets agents revise toward each other. Moderate conformity plus genuine
reconsideration beats one-shot voting; heavy conformity herds the committee
onto the early plurality and the gain evaporates.

In [ ]:
p, n = 0.58, 9
committee = Committee(n_agents=n, n_answers=4, accuracy=p, correlation=0.0, seed=0)
one_shot = committee_accuracy(committee, 3000, 1)
conformities = np.linspace(0, 1, 11)
finals = []
for k in conformities:
    rng = np.random.default_rng(1)
    finals.append(debate(committee, 3000, rng, rounds=5, conformity=float(k))[-1])

plt.plot(conformities, finals, "o-", color="#c44e52", label="debate (final)")
plt.axhline(one_shot, color="#55a467", ls="--", label=f"one-shot vote ({one_shot:.2f})")
plt.axhline(p, color="#888", ls=":", label=f"single agent ({p})")
plt.xlabel("conformity"); plt.ylabel("accuracy"); plt.legend()
plt.grid(alpha=0.3); plt.show()

The peak at moderate conformity is the sweet spot; the decline toward κ=1 is
herding — the committee agrees before the evidence is in.

## 4. Pipelines compound errors

Agents in series: every stage must succeed, so reliability is p^L — the same
cliff as long-horizon tool use (Post 3b).

In [ ]:
stages = np.arange(1, 13)
for p, c in [(0.95, "#55a467"), (0.9, "#3a7ebf"), (0.8, "#dd8452"), (0.7, "#c44e52")]:
    plt.plot(stages, [pipeline_accuracy(p, L) for L in stages], "o-",
             color=c, label=f"per-stage {p}")
plt.axhline(0.5, color="#bbb", ls=":")
plt.xlabel("pipeline stages"); plt.ylabel("end-to-end success"); plt.legend()
plt.grid(alpha=0.3); plt.show()
print("Ten 0.95-reliable stages:", round(pipeline_accuracy(0.95, 10), 2))

## What's next

You've seen the forces that govern multi-agent systems:

- **Condorcet** — independent agents above chance vote toward certainty.
- **Correlation caps it** — shared base models and prompts collapse the
  effective committee size; diversity is the resource.
- **Debate** — peer critique helps with reconsideration, herds under
  conformity; a confident wrong agent poisons the group (calibration again).
- **Pipelines** — reliability compounds as p^L; specialize only when the
  skill edge beats the handoff tax.

This closes the conceptual arc. The final post — **Post 5b: the nanoAgent
reference implementation** — wires the whole curriculum (planning, tools,
retrieval, reflection, calibration, coordination) into a small, readable agent
you can run end to end.